In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_3")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

In [ ]:
from plotly import graph_objs as go

# sankey figure of where tracklets go.

df = spots_dfs[0].copy()
print(df["status"].unique())

df = df.query("AP < 0.98 and status != 6")

df["fallout"] = df["distance_from_surface"] < -8

t = df.groupby("tracklet_id").agg({
    "cycle": "first",
    "fallout": "last",
    "id_kept": "last",
    "parent_id": "first",
    "track_id": "first",
    "frame": "count",
})

t["parent_tracklet"] = t["parent_id"].map(df["tracklet_id"])
t["num_children"] = t.index.map(t["parent_tracklet"].value_counts()).fillna(0).astype(int)
print(t.groupby("num_children")["fallout"].count())

print(t)

fallout_tracklets = t[t["fallout"]].index
no_children_tracklets = t[t["num_children"] == 0].index
started_fallout_tracklets = t[df.groupby("tracklet_id")["fallout"].first() == 1].index
no_parent_late_tracklets = t[(t["parent_id"] == -1) & (t["cycle"] >= 12)].index
parent_one_child_tracklets = t[t["parent_tracklet"].map(t["num_children"]) == 1].index

df_fallout = df.query("tracklet_id.isin(@fallout_tracklets) and "
                      "tracklet_id.isin(@no_children_tracklets) and "
                      "AP < 0.97 and "
                      "not tracklet_id.isin(@started_fallout_tracklets) and "
                      "not tracklet_id.isin(@no_parent_late_tracklets) and "
                      "not tracklet_id.isin(@parent_one_child_tracklets)").copy()

# print(len(df_fallout["tracklet_id"].unique() / len()))

t_fallout_frame = df_fallout.groupby("tracklet_id")["fallout"].idxmax().map(df_fallout["time_since_nc11"])
t_fallout_ap = df_fallout.groupby("tracklet_id")["fallout"].idxmax().map(df_fallout["AP"])
t_fallout_cycle = df_fallout.groupby("tracklet_id")["fallout"].idxmax().map(df_fallout["cycle"])

t_arrival = df.groupby("track_id")["time_since_nc11"].min()
valid = (t_arrival < 5) * (df.groupby("track_id")["frame"].min() > df["frame"].min())
t_arrival = t_arrival[valid]
t_AP = df.groupby("track_id")["AP"].first()[valid]

fig, ax = plt.subplots(1, 1, figsize=(3, 2.25))
sns.scatterplot(y=t_fallout_frame, x=t_fallout_ap, color="#0a9396", edgecolor="k", )
sns.scatterplot(y=t_arrival, x=t_AP, color="#ee9b00", edgecolor="k", )
ax.spines[["top", "right"]].set_visible(False)

plt.xlabel("AP position")
plt.ylabel("Time since NC11 (min)")
plt.savefig(save_path / "fallout_and_arrival_by_ap.png", dpi=300, bbox_inches="tight")
plt.show()
print()



print(t)

df_filtered = df.query("AP < 0.97 and "
                      "not tracklet_id.isin(@started_fallout_tracklets) and "
                      "not tracklet_id.isin(@no_parent_late_tracklets) and "
                      "not tracklet_id.isin(@parent_one_child_tracklets)").copy()

print(df_filtered.groupby("tracklet_id")["fallout"].any().mean())

t_fallout = df_filtered.groupby("tracklet_id")["fallout"].any()
t_cycle = df_filtered.groupby("tracklet_id")["cycle"].last()
t_first = df_filtered.groupby("tracklet_id")["parent_id"].first() == -1


print(t_cycle.value_counts())
print(t_fallout.groupby(t_cycle).mean())
print(t_first.groupby(t_cycle).sum())

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
sns.barplot(x=t_cycle, y=t_fallout, errorbar=None, edgecolor="k", color="#0a9396")
plt.xlabel("Cycle")
plt.ylabel("Rate of fallout")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / "fallout_by_cycle.png", dpi=300, bbox_inches="tight")
plt.show()

df_filtered = df.query("AP < 0.97 and "
                      "not tracklet_id.isin(@no_parent_late_tracklets) and "
                      "not tracklet_id.isin(@parent_one_child_tracklets)").copy()

track_id_first_cycle = df_filtered.groupby("track_id")["cycle"].first()
track_id_first_frame = df_filtered.groupby("track_id")["frame"].first()
track_id_before_12 = track_id_first_cycle[track_id_first_cycle < 12].index
# sns.barplot(df.query("frame == frame.max() and track_id.isin(@track_id_before_12)").groupby("track_id")["frame"].count().value_counts())

# sns.barplot(df.query("frame == frame.max() and track_id.isin(@track_id_before_12)").groupby("track_id")["frame"].count().value_counts())
df2 = df_filtered.query("frame == frame.max() and track_id.isin(@track_id_before_12)")
df2["first_cycle"] = df2["track_id"].map(track_id_first_cycle)
df2["first_frame"] = df2["track_id"].map(track_id_first_frame)
t = df2.groupby("track_id").agg({
    "frame": "count",
    "first_cycle": "first",
    "first_frame": "first",
})

t["cycle"] = (t["first_frame"] > 26) + 10
t = t[t["cycle"] == 10]

sns.histplot(t["first_frame"])
print(t["first_frame"].value_counts())
print(t["frame"].value_counts())
plt.show()




# x = t.reset_index().groupby(["frame", "first_cycle"])["track_id"].count().reset_index()
print(t)
fig, ax = plt.subplots(1, 1, figsize=(1.8, 1.8))
sns.histplot(t, x="frame", hue="cycle", binwidth=1.0, palette={10:"#ee9b00", 11: "#ae2012"}, multiple="stack", hue_order=[11, 10], edgecolor="k", binrange=(0.5, 16.5), stat="probability", alpha=1.0, legend=False)
ax = plt.gca()
ax.spines[["top", "right"]].set_visible(False)
plt.xlabel("# of nuclei in NC14")
plt.xticks([0, 4, 8, 12, 16])
plt.ylabel("Fraction of lineages")
# plt.legend(title="Lineage arrival", labels=["NC10", "NC11"])
plt.savefig(save_path / "lineage_arrival_by_cycle.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
sns.histplot(t, x="frame", hue="cycle", binwidth=1.0, palette={10:"#ee9b00", 11: "#ae2012"}, multiple="stack", hue_order=[11, 10], edgecolor="k", binrange=(0.5, 16.5))
ax = plt.gca()
ax.spines[["top", "right"]].set_visible(False)
plt.xlabel("# of nuclei in NC14")
plt.xticks([0, 4, 8, 12, 16])
plt.ylabel("Number of lineages")
plt.yscale("log")
plt.legend(title="Lineage arrival", labels=["NC10", "NC11"])
plt.savefig(save_path / "fraction_lineage_arrival_by_cycle.png", dpi=300, bbox_inches="tight")
plt.show()
# sns.histplot(df, x="distance_from_surface")
#
# import napari
# viewer = napari.Viewer()
# color = ["red" if dis < -8 else "blue" for dis in df["distance_from_surface"]]
# viewer.add_points(df[["frame", "z", "y", "x"]].values, size=2*df["radius"], face_color=color)

In [ ]:
from scipy.spatial import KDTree
from itertools import pairwise
from collections import defaultdict

displacements = defaultdict(list)

for left, right in pairwise([0.0, 0.2, 0.4, 0.6, 0.8, 1.0]):
    for k, df in enumerate(spots_dfs):
        df = df[df["AP"].between(left, right)].copy()
        df["child_id"] = df.index.map({
            parent: child for parent, child in zip(df["parent_id"], df.index)
        })
        for metric in ["dz", "dy", "dx"]:
            df[f"child_{metric}"] = df["child_id"].map(df[metric])
        cycle = 11
        frame_start = int(df[df["cycle"] == cycle].groupby("tracklet_id")["frame"].min().quantile(0.5))
        frame_end = int(df[df["cycle"] == cycle].groupby("tracklet_id")["frame"].max().quantile(0.5))

        dis, dx = [], []

        for frame in range(frame_start+5, frame_end-5):
            frame_df = df[df["frame"] == frame].copy()
            points = frame_df[["z", "y", "x"]].values

            changes = frame_df[["child_dz", "child_dy", "child_dx"]].values
            tree = KDTree(points)
            dists, ii = tree.query(points, k=2)
            dis.extend(dists[:, 1:].flatten())

            a_index = np.arange(ii.shape[0])[:, None]

            relative_position_vectors = points[:, None, :] - points[ii[:, 1:], :]
            relative_movement_vectors = changes[:, None, :] - changes[ii[:, 1:], :]

            normalized_relative_position_vectors = relative_position_vectors / np.linalg.norm(relative_position_vectors, axis=-1, keepdims=True)
            dx_values = np.einsum("ijk, ijk -> ij", normalized_relative_position_vectors, relative_movement_vectors)
            dx.extend(dx_values.flatten())

        displacements["location"].append(left)
        displacements["distance"].append(np.nanmean(dis))
        displacements["dx"].append(np.nanmean(dx))
        displacements["source"].append(k)
        condition = condition_map[stems[k][:-6]]
        displacements["condition"].append(condition)
        # plt.scatter(dis, dx, s=0.3, alpha=0.5, color="r")
        # sns.lineplot(x=(np.array(dis) // 0.5) * 0.5 , y=dx, errorbar=None, label=left)
displacements_df = pd.DataFrame(displacements)


In [ ]:
sns.barplot(
    data=displacements_df[displacements_df["condition"].isin(["wt", "bcd"])], x="location", y="dx", hue="condition",
    palette=condition_main_colors, errorbar=None)
sns.stripplot(
    data=displacements_df[displacements_df["condition"].isin(["wt", "bcd"])], x="location", y="dx", hue="condition",
    palette="dark:k", dodge=True, alpha=0.5, size=5)

plt.show()

sns.lineplot(
    data=displacements_df[displacements_df["condition"].isin(["wt", "bcd"])], x="location", y="distance", hue="condition", style="source", dashes=False,
    palette=condition_main_colors, errorbar=None, legend=False)
plt.show()